In [1]:
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib

# add path to project root
import sys
PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT))
if PROJECT_ROOT.name == "notebooks":
    sys.path.append(str(PROJECT_ROOT.parent))
    
    
from src.preprocess.gaming_text_preprocessor import GamingTextPreprocessor
from src.preprocess.gaming_text_labeler import GamingTextLabeler
from src.ensemble import ensemble


# instantiate preprocessor and labeler
gaming_preprocessor = GamingTextPreprocessor()
gaming_labeler = GamingTextLabeler()

In [2]:
# config
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data' / 'processed_data').exists() and PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG = {
    'seed': 7524,
    'data_dir': PROJECT_ROOT / 'data' / 'processed_data',
    'model_dir': PROJECT_ROOT / 'models',
    'text_col': 'message',
    'label_col': 'label'
}

np.random.seed(CONFIG['seed'])
seed = CONFIG['seed']
tc, lc = CONFIG['text_col'], CONFIG['label_col']

print('CONFIG loaded. Text column:', tc)


CONFIG loaded. Text column: message


# Fit the Gaming Text Preprocessor 

In [3]:
# load WOT train data
d = CONFIG['data_dir']

wot_train = pd.read_parquet(d / 'wot' / 'x_train.parquet')
wot_train["data_source"] = "wot"

In [4]:
# load DOTA training data
dota_train = pd.read_parquet(d / 'dota' / 'x_train.parquet')
dota_train["data_source"] = "dota"

In [5]:
# combine datasets
train = pd.concat([wot_train, dota_train], ignore_index=True)

In [6]:
# train 
gaming_preprocessor.fit(train, text_column=tc)

# Label Cleaning

In [7]:
# convert dota labels 

train = gaming_labeler.convert_three_class(
    train, 
    label_column=lc, 
    data_source_column='data_source',
    output_column='label_3class'
)


In [8]:
train["label_3class"].value_counts()

label_3class
0    40929
1     7492
2     5286
Name: count, dtype: int64

# Model Loading

In [9]:
# load all models from disk
model_dir = CONFIG['model_dir'] / "multi" 
model_paths = list(model_dir.glob("*.joblib"))
models = [joblib.load(path) for path in model_paths]

models

[{'pipeline': Pipeline(steps=[('tfidf',
                   TfidfVectorizer(max_df=0.95, ngram_range=(1, 2),
                                   sublinear_tf=True, token_pattern=None,
                                   tokenizer=<function tokenize at 0x118e763e0>)),
                  ('clf',
                   LinearSVC(C=0.7113292323780791, class_weight='balanced',
                             dual=True, loss='hinge', max_iter=2000,
                             random_state=7524, tol=0.005851894281015725))]),
  'model_name': 'LinearSVC',
  'cv_row': {'Model': 'LinearSVC',
   'cv_f1': 0.7875,
   'cv_f1_std': 0.0054,
   'cv_recall': 0.7739,
   'cv_precision': 0.8029},
  'game': 'Combined',
  'label_scheme': {0: 0, 1: 2, 2: 2, 3: 1},
  'seed': 7524},
 {'pipeline': Pipeline(steps=[('tfidf',
                   TfidfVectorizer(max_df=0.95, ngram_range=(1, 2),
                                   sublinear_tf=True, token_pattern=None,
                                   tokenizer=<function tokeni

In [10]:
# instantiate the ensemble with model paths
multiclass_ensemble = ensemble.Ensemble(
    classifiers_paths=model_paths
)

In [11]:

X_train_text = gaming_preprocessor.transform(train, text_column=tc)

In [12]:
# run simple majority voting on the training data
multiclass_ensemble.predict_simple_majority(X_train_text)

Predicting with SimpleMajority...


NotFittedError: This LinearSVC instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.